<a href="https://colab.research.google.com/github/Raman676/LLM-based-floor-plan-generator/blob/main/LLM_based_floor_plan_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-huggingface langgraph pydantic huggingface-hub
print("✅ All dependencies installed successfully!")

✅ All dependencies installed successfully!


In [ ]:
import os
from getpass import getpass

HUGGINGFACE_TOKEN = getpass("Enter your HuggingFace API Token: ")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACE_TOKEN
print("✅ API Token set successfully!")

Enter your HuggingFace API Token: ··········
✅ API Token set successfully!


In [ ]:
import os
from getpass import getpass

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List, Dict
from langchain.tools import tool

# For structured output
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

import json

print("📦 Libraries imported successfully!")

print("🤖 Initializing model...")

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",  # Using an open-source model
    task="text-generation",
    max_new_tokens=2048,
    temperature=0.3,
)

model = ChatHuggingFace(llm=llm)

print("✅ Model initialized!")

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",  # Using a reliable open-source model
    task="text-generation",
    max_new_tokens=2048,
    temperature=0.3,
)

model = ChatHuggingFace(llm=llm)

@tool
def cal_area(width: int, height: int) -> int:
    """
    Calculate the area of a room.

    Args:
        width: Width of the room in inches
        height: Height/length of the room in inches

    Returns:
        Area of the room in square inches
    """
    return width * height

@tool
def sum_numbers(numbers: list[float]) -> float:
    """
    Calculate the total area by summing individual room areas.

    Use this tool when you need to find the total area of multiple rooms.
    First calculate each room's area separately, then pass all the areas
    as a list to get the total combined area.

    Args:
        numbers: A list of individual room areas in square inches

    Returns:
        Total combined area of all rooms in square inches
    """
    return sum(numbers)

tools = [cal_area, sum_numbers]
model_with_tools = model.bind_tools(tools)
# ==================================================================================
# SCHEMA DEFINITIONS
# ==================================================================================

class Room(BaseModel):
    rtype: str = Field(description="Type of room (bedroom, kitchen, bathroom, living_room, hall, etc.)")
    id: int = Field(description="Unique identifier for the room, starting from 1")
    width: int = Field(description="Width of the room in inches")
    height: int = Field(description="Height/length of the room in inches")
    position: List[int] = Field(description="Position [x, y] coordinates of the room")
    windows: int = Field(description="Number of windows in the room")
    doors: int = Field(description="Number of doors in the room")

class Connection(BaseModel):
    from_room: int = Field(alias="from", description="id of the source room")
    to_room: int = Field(alias="to", description="id of the destination room")
    rtype: str = Field(description="Type of connection: 'door' or 'window'")

class FloorPlanSchema(BaseModel):
    rooms: List[Room] = Field(description="List of all rooms in the floor plan")
    connections: List[Connection] = Field(description="List of connections between rooms")
    total_area: int = Field(description="Total area of the floor plan in square inches")

# Parser for structured output
parser = PydanticOutputParser(pydantic_object=FloorPlanSchema)
instructions = parser.get_format_instructions()

# ==================================================================================

class room_scheme(BaseModel):
    rtype: str = Field(description="Type of room (bedroom, kitchen, bathroom, living_room, hall, etc.)")
    id: int = Field(description="Unique identifier for the room, starting from 1")
    width: int = Field(description="Width of the room in inches")
    height: int = Field(description="Height/length of the room in inches")
    windows: int = Field(default=0, description="Number of windows in the room")
    doors: int = Field(default=1, description="Number of doors in the room")


class getrooms(BaseModel):
    rooms: List[room_scheme] = Field(description="List of all rooms in the floor plan")

parser_r = PydanticOutputParser(pydantic_object=getrooms)
instructions_r = parser_r.get_format_instructions()

# ==================================================================================

class getconnections(BaseModel):
    connections: List[Connection] = Field(description="List of all Connections connecting different rooms in the floor plan")

parser_c = PydanticOutputParser(pydantic_object=getconnections)
instructions_c = parser_c.get_format_instructions()

# ==================================================================================

class PositionedRooms(BaseModel):
    rooms: List[Room] = Field(description="List of rooms with assigned [x, y] positions")

parser_p = PydanticOutputParser(pydantic_object=PositionedRooms)
instructions_p = parser_p.get_format_instructions()

# ==================================================================================



# ==================================================================================
room_extraction_prompt = PromptTemplate(
    input_variables=["user_input"],
    template="""You are an expert floor plan analyzer. Extract all room information from the user's description.

User Input: {user_input}

Output Instructions: {instructions_r}

YOUR RESPONSE MUST CONTAIN ONLY THE JSON OBJECT AND NOTHING ELSE. DO NOT INCLUDE ANY CONVERSATIONAL TEXT OR EXPLANATIONS. Output the JSON object wrapped in markdown code delimiters, e.g., ```json{{\"rooms\":[...]}}```.

Extract the following for EACH room mentioned:
- Room type (bedroom, kitchen, bathroom, living_room, hall, etc.)
- Dimensions (width and height in inches)
- Number of windows (default: 1 if not specified)
- Number of doors (default: 1 if not specified)

Default dimensions if not specified:
- Bedroom: 180x144
- Kitchen: 120x120
- Bathroom: 96x72
- Living room: 200x180
- Hall: 96x60

Output ONLY with this structure (a JSON object with a 'rooms' key containing a list of rooms, wrapped in ```json tags):
```json
{{
  "rooms": [
    {{
      "rtype": "bedroom",
      "id": 1,
      "width": 180,
      "height": 144,
      "windows": 2,
      "doors": 1
    }},
    {{
      "rtype": "kitchen",
      "id": 2,
      "width": 120,
      "height": 120,
      "windows": 1,
      "doors": 1
    }},
    {{
      "rtype": "bathroom",
      "id": 3,
      "width": 96,
      "height": 72,
      "windows": 1,
      "doors": 1
    }}
  ]
}}
```
Output:```json
""",
    partial_variables={"instructions_r": instructions_r}
)

connection_extraction_prompt = PromptTemplate(
    input_variables=["user_input", "rooms"],
    template="""You are an expert floor plan analyzer. Extract all connections between rooms.

User Input: {user_input}

Available Rooms: {rooms}



Identify all relationships:
- "Room A connects to Room B" → door connection
- "Room A is adjacent to Room B" → door connection
- "Room A has a window facing Room B" → window connection
- "Room A has windows" → window connection to outside (from: room_id, to: room_id - same ID)

Output Instructions: {instructions_c}

Output:""",
    partial_variables={"instructions_c": instructions_c}
)

class PositionedRooms(BaseModel):
    rooms: List[Room] = Field(description="List of rooms with assigned [x, y] positions")

parser_p = PydanticOutputParser(pydantic_object=PositionedRooms)
instructions_p = parser_p.get_format_instructions()

positioning_prompt = PromptTemplate(
    input_variables=["extracted_rooms", "extracted_connections"],
    template="""You are an expert floor plan designer. Assign [x, y] coordinates to each room in the floor plan.
Consider the dimensions of each room and the connections between them to place them logically adjacent to each other.
Assume the origin [0,0] is the bottom-left corner of the overall floor plan.
Ensure rooms connected by a 'door' are adjacent. Windows can face outside, so they don't necessarily imply adjacency.

Extracted Rooms: {extracted_rooms}
Extracted Connections: {extracted_connections}

Output Instructions: {instructions_p}

""",
    partial_variables={"instructions_p": instructions_p}
)


# ==================================================================================
# STATE DEFINITION
# ==================================================================================

class FloorPlanState(TypedDict):
    user_input: str
    extracted_rooms: List[Dict]
    extracted_connections: List[Dict]
    positioned_rooms: List[Dict]
    final_output: Dict

# ==================================================================================
# NODE FUNCTIONS
# ==================================================================================
 # 1 - Room Extraction

def extract_rooms(state: FloorPlanState) -> Dict:
    """Extract room details from natural language input"""
    print("Extracting rooms...")
    chain = room_extraction_prompt | model | parser_r
    print("Chain is Ready (raw output mode)")
    llm_response = chain.invoke({"user_input": state["user_input"]}) # llm_response is a getrooms object
    print("Response Invoked")

    # The llm_response is already a parsed Pydantic object (getrooms)
    # We need to extract the list of room_scheme objects from it.
    content = [room.model_dump() for room in llm_response.rooms]

    state["extracted_rooms"] = content if content else []

    if state["extracted_rooms"]:
        print(f"✅ Rooms extracted successfully! Found {len(state['extracted_rooms'])} rooms")
    else:
        print("❌ No rooms found!")

    return state


 # 2 - Connection Extraction

def extract_connections(state: FloorPlanState) -> Dict:
    """Extract connections between rooms"""
    print("🔗 Extracting connections...")
    chain = connection_extraction_prompt | model | parser_c
    response = chain.invoke({
        "user_input": state["user_input"],
        "rooms": json.dumps(state["extracted_rooms"])
    })

    # Converting Pydantic objects to dicts
    content = [conn.model_dump(by_alias=True) for conn in response.connections]
    state["extracted_connections"] = content if content else []

    if state["extracted_connections"]:
        print(f"✅ Connections extracted successfully! Found {len(state['extracted_connections'])} connections")
    else:
        print("❌ No connections found!")

    return state


    # 3 - Room Positioning

def position_rooms(state: FloorPlanState) -> Dict:
    """Generate positions for the extracted rooms"""
    print("Positioning rooms...")
    chain = positioning_prompt | model | parser_p
    response = chain.invoke({
        "extracted_rooms": json.dumps(state["extracted_rooms"]),
        "extracted_connections": json.dumps(state["extracted_connections"])
    })


    state["positioned_rooms"] = [room.model_dump() for room in response.rooms]

    if state["positioned_rooms"]:
        print(f"✅ Rooms positioned successfully! Positioned {len(state['positioned_rooms'])} rooms")
    else:
        print("❌ No rooms positioned!")

    return state

# 4 - Final Output Formatting

def format_final_output(state: FloorPlanState) -> Dict:
    """Formats the extracted and positioned data into the final FloorPlanSchema."""
    print("Formatting final output...")

    # ===================
    Areas = []
    for room in state["positioned_rooms"]:
        area = room["width"] * room["height"]
        Areas.append(area)

    Total_Area = sum(Areas)



    formatted_rooms = [Room(**room_data) for room_data in state["positioned_rooms"]]


    formatted_connections = [Connection(**conn) for conn in state["extracted_connections"]]

    final_floor_plan = FloorPlanSchema(
        rooms=formatted_rooms,
        connections=formatted_connections,
        total_area=Total_Area

    )

    state["final_output"] = final_floor_plan.model_dump(by_alias=True)
    print("✅ Final output formatted successfully!")
    return state

# ==================================================================================
# WORKFLOW DEFINITION
# ==================================================================================

workflow = StateGraph(FloorPlanState)

workflow.add_node("extract_rooms", extract_rooms)
workflow.add_node("extract_connections", extract_connections)
workflow.add_node("position_rooms", position_rooms)
workflow.add_node("format_final_output", format_final_output)

workflow.set_entry_point("extract_rooms")

workflow.add_edge("extract_rooms", "extract_connections")
workflow.add_edge("extract_connections", "position_rooms")
workflow.add_edge("position_rooms", "format_final_output")
workflow.set_finish_point("format_final_output")

app = workflow.compile()

📦 Libraries imported successfully!
🤖 Initializing model...
✅ Model initialized!


  best model

In [ ]:
import os
from getpass import getpass

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List, Dict
from langchain.tools import tool

# For structured output
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

import json

print("📦 Libraries imported successfully!")

print("🤖 Initializing model...")

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",  # Using an open-source model
    task="text-generation",
    max_new_tokens=2048,
    temperature=0.3,
)

model = ChatHuggingFace(llm=llm)

print("✅ Model initialized!")

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",  # Using a reliable open-source model
    task="text-generation",
    max_new_tokens=2048,
    temperature=0.3,
)

model = ChatHuggingFace(llm=llm)

@tool
def cal_area(width: int, height: int) -> int:
    """
    Calculate the area of a room.

    Args:
        width: Width of the room in inches
        height: Height/length of the room in inches

    Returns:
        Area of the room in square inches
    """
    return width * height

@tool
def sum_numbers(numbers: list[float]) -> float:
    """
    Calculate the total area by summing individual room areas.

    Use this tool when you need to find the total area of multiple rooms.
    First calculate each room's area separately, then pass all the areas
    as a list to get the total combined area.

    Args:
        numbers: A list of individual room areas in square inches

    Returns:
        Total combined area of all rooms in square inches
    """
    return sum(numbers)

tools = [cal_area, sum_numbers]
model_with_tools = model.bind_tools(tools)
# ==================================================================================
# SCHEMA DEFINITIONS
# ==================================================================================

class Room(BaseModel):
    rtype: str = Field(description="Type of room (bedroom, kitchen, bathroom, living_room, hall, etc.)")
    id: int = Field(description="Unique identifier for the room, starting from 1")
    width: int = Field(description="Width of the room in inches")
    height: int = Field(description="Height/length of the room in inches")
    position: List[int] = Field(description="Position [x, y] coordinates of the room")
    windows: int = Field(description="Number of windows in the room")
    doors: int = Field(description="Number of doors in the room")

class Connection(BaseModel):
    from_room: int = Field(alias="from", description="id of the source room")
    to_room: int = Field(alias="to", description="id of the destination room")
    rtype: str = Field(description="Type of connection: 'door' or 'window'")

class FloorPlanSchema(BaseModel):
    rooms: List[Room] = Field(description="List of all rooms in the floor plan")
    connections: List[Connection] = Field(description="List of connections between rooms")
    total_area: int = Field(description="Total area of the floor plan in square inches")

# Parser for structured output
parser = PydanticOutputParser(pydantic_object=FloorPlanSchema)
instructions = parser.get_format_instructions()

# ==================================================================================

class room_scheme(BaseModel):
    rtype: str = Field(description="Type of room (bedroom, kitchen, bathroom, living_room, hall, etc.)")
    id: int = Field(description="Unique identifier for the room, starting from 1")
    width: int = Field(description="Width of the room in inches")
    height: int = Field(description="Height/length of the room in inches")
    windows: int = Field(default=0, description="Number of windows in the room")
    doors: int = Field(default=1, description="Number of doors in the room")


class getrooms(BaseModel):
    rooms: List[room_scheme] = Field(description="List of all rooms in the floor plan")

parser_r = PydanticOutputParser(pydantic_object=getrooms)
instructions_r = parser_r.get_format_instructions()

# ==================================================================================

class getconnections(BaseModel):
    connections: List[Connection] = Field(description="List of all Connections connecting different rooms in the floor plan")

parser_c = PydanticOutputParser(pydantic_object=getconnections)
instructions_c = parser_c.get_format_instructions()

# ==================================================================================

class PositionedRooms(BaseModel):
    rooms: List[Room] = Field(description="List of rooms with assigned [x, y] positions")

parser_p = PydanticOutputParser(pydantic_object=PositionedRooms)
instructions_p = parser_p.get_format_instructions()

# ==================================================================================



# ==================================================================================
room_extraction_prompt = PromptTemplate(
    input_variables=["user_input"],
    template="""You are an expert floor plan analyzer. Extract all room information from the user's description.

User Input: {user_input}

Output Instructions: {instructions_r}

YOUR RESPONSE MUST CONTAIN ONLY THE JSON OBJECT AND NOTHING ELSE. DO NOT INCLUDE ANY CONVERSATIONAL TEXT OR EXPLANATIONS. Output the JSON object wrapped in markdown code delimiters, e.g., ```json{{\"rooms\":[...]}}```.

Extract the following for EACH room mentioned:
- Room type (bedroom, kitchen, bathroom, living_room, hall, etc.)
- Dimensions (width and height in inches)
- Number of windows (default: 1 if not specified)
- Number of doors (default: 1 if not specified)

Default dimensions if not specified:
- Bedroom: 180x144
- Kitchen: 120x120
- Bathroom: 96x72
- Living room: 200x180
- Hall: 96x60

Output ONLY with this structure (a JSON object with a 'rooms' key containing a list of rooms, wrapped in ```json tags):
```json
{{
  "rooms": [
    {{
      "rtype": "bedroom",
      "id": 1,
      "width": 180,
      "height": 144,
      "windows": 2,
      "doors": 1
    }},
    {{
      "rtype": "kitchen",
      "id": 2,
      "width": 120,
      "height": 120,
      "windows": 1,
      "doors": 1
    }},
    {{
      "rtype": "bathroom",
      "id": 3,
      "width": 96,
      "height": 72,
      "windows": 1,
      "doors": 1
    }}
  ]
}}
```
Output:```json
""",
    partial_variables={"instructions_r": instructions_r}
)

connection_extraction_prompt = PromptTemplate(
    input_variables=["user_input", "rooms"],
    template="""You are an expert floor plan analyzer. Extract all connections between rooms.

User Input: {user_input}

Available Rooms: {rooms}



Identify all relationships:
- "Room A connects to Room B" → door connection
- "Room A is adjacent to Room B" → door connection
- "Room A has a window facing Room B" → window connection
- "Room A has windows" → window connection to outside (from: room_id, to: room_id - same ID)

Output Instructions: {instructions_c}

Output:""",
    partial_variables={"instructions_c": instructions_c}
)

class PositionedRooms(BaseModel):
    rooms: List[Room] = Field(description="List of rooms with assigned [x, y] positions")

parser_p = PydanticOutputParser(pydantic_object=PositionedRooms)
instructions_p = parser_p.get_format_instructions()

positioning_prompt = PromptTemplate(
    input_variables=["extracted_rooms", "extracted_connections"],
    template="""You are an expert floor plan designer. Assign [x, y] coordinates to each room in the floor plan.
Consider the dimensions of each room and the connections between them to place them logically adjacent to each other.
Assume the origin [0,0] is the bottom-left corner of the overall floor plan.
Ensure rooms connected by a 'door' are adjacent. Windows can face outside, so they don't necessarily imply adjacency.

Extracted Rooms: {extracted_rooms}
Extracted Connections: {extracted_connections}

Output Instructions: {instructions_p}

""",
    partial_variables={"instructions_p": instructions_p}
)


# ==================================================================================
# STATE DEFINITION
# ==================================================================================

class FloorPlanState(TypedDict):
    user_input: str
    extracted_rooms: List[Dict]
    extracted_connections: List[Dict]
    positioned_rooms: List[Dict]
    final_output: Dict

# ==================================================================================
# NODE FUNCTIONS
# ==================================================================================
 # 1 - Room Extraction

def extract_rooms(state: FloorPlanState) -> Dict:
    """Extract room details from natural language input"""
    print("Extracting rooms...")
    chain = room_extraction_prompt | model | parser_r
    print("Chain is Ready (raw output mode)")
    llm_response = chain.invoke({"user_input": state["user_input"]}) # llm_response is a getrooms object
    print("Response Invoked")

    # The llm_response is already a parsed Pydantic object (getrooms)
    # We need to extract the list of room_scheme objects from it.
    content = [room.model_dump() for room in llm_response.rooms]

    state["extracted_rooms"] = content if content else []

    if state["extracted_rooms"]:
        print(f"✅ Rooms extracted successfully! Found {len(state['extracted_rooms'])} rooms")
    else:
        print("❌ No rooms found!")

    return state


 # 2 - Connection Extraction

def extract_connections(state: FloorPlanState) -> Dict:
    """Extract connections between rooms"""
    print("🔗 Extracting connections...")
    chain = connection_extraction_prompt | model | parser_c
    response = chain.invoke({
        "user_input": state["user_input"],
        "rooms": json.dumps(state["extracted_rooms"])
    })

    # Converting Pydantic objects to dicts
    content = [conn.model_dump(by_alias=True) for conn in response.connections]
    state["extracted_connections"] = content if content else []

    if state["extracted_connections"]:
        print(f"✅ Connections extracted successfully! Found {len(state['extracted_connections'])} connections")
    else:
        print("❌ No connections found!")

    return state


    # 3 - Room Positioning

def position_rooms(state: FloorPlanState) -> Dict:
    """Generate positions for the extracted rooms"""
    print("Positioning rooms...")
    chain = positioning_prompt | model | parser_p
    response = chain.invoke({
        "extracted_rooms": json.dumps(state["extracted_rooms"]),
        "extracted_connections": json.dumps(state["extracted_connections"])
    })


    state["positioned_rooms"] = [room.model_dump() for room in response.rooms]

    if state["positioned_rooms"]:
        print(f"✅ Rooms positioned successfully! Positioned {len(state['positioned_rooms'])} rooms")
    else:
        print("❌ No rooms positioned!")

    return state

# 4 - Final Output Formatting

def format_final_output(state: FloorPlanState) -> Dict:
    """Formats the extracted and positioned data into the final FloorPlanSchema."""
    print("Formatting final output...")

    # ===================
    Areas = []
    for room in state["positioned_rooms"]:
        area = room["width"] * room["height"]
        Areas.append(area)

    Total_Area = sum(Areas)



    formatted_rooms = [Room(**room_data) for room_data in state["positioned_rooms"]]


    formatted_connections = [Connection(**conn) for conn in state["extracted_connections"]]

    final_floor_plan = FloorPlanSchema(
        rooms=formatted_rooms,
        connections=formatted_connections,
        total_area=Total_Area

    )

    state["final_output"] = final_floor_plan.model_dump(by_alias=True)
    print("✅ Final output formatted successfully!")
    return state

# ==================================================================================
# WORKFLOW DEFINITION
# ==================================================================================

workflow = StateGraph(FloorPlanState)

workflow.add_node("extract_rooms", extract_rooms)
workflow.add_node("extract_connections", extract_connections)
workflow.add_node("position_rooms", position_rooms)
workflow.add_node("format_final_output", format_final_output)

workflow.set_entry_point("extract_rooms")

workflow.add_edge("extract_rooms", "extract_connections")
workflow.add_edge("extract_connections", "position_rooms")
workflow.add_edge("position_rooms", "format_final_output")
workflow.set_finish_point("format_final_output")

app = workflow.compile()

📦 Libraries imported successfully!
🤖 Initializing model...
✅ Model initialized!


In [ ]:
# ===================== IMPORTS =====================
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langgraph.graph import StateGraph
from typing import TypedDict, List, Dict
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
import json

# ===================== MODEL =====================
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    max_new_tokens=2048,
    temperature=0.3,
)
model = ChatHuggingFace(llm=llm)

# ===================== FURNITURE DEFINITIONS =====================
FURNITURE_LIBRARY = {
    "wardrobe": {"width": 60, "height": 24},
    "sink": {"width": 24, "height": 18},
}

# ===================== SCHEMAS =====================
class Furniture(BaseModel):
    ftype: str
    width: int
    height: int
    position: List[int]

class Room(BaseModel):
    rtype: str
    id: int
    width: int
    height: int
    position: List[int]
    windows: int
    doors: int
    furniture: List[Furniture] = []

class Connection(BaseModel):
    from_room: int = Field(alias="from")
    to_room: int = Field(alias="to")
    rtype: str

class FloorPlanSchema(BaseModel):
    rooms: List[Room]
    connections: List[Connection]
    total_area: int

# New Pydantic model to wrap the list of rooms
class RoomsContainer(BaseModel):
    rooms: List[Room]

# Parser and instructions for RoomsContainer
parser_rooms_container = PydanticOutputParser(pydantic_object=RoomsContainer)
instructions_rooms_container = parser_rooms_container.get_format_instructions()

# ===================== PROMPTS =====================
room_prompt = PromptTemplate(
    input_variables=["user_input"],
    template="""You are an expert floor plan analyzer. Extract all room information from the user's description.

User Input: {user_input}

Output Instructions: {instructions_rooms_container}

YOUR RESPONSE MUST CONTAIN ONLY THE JSON OBJECT AND NOTHING ELSE. DO NOT INCLUDE ANY CONVERSATIONAL TEXT OR EXPLANATIONS. Output the JSON object wrapped in markdown code delimiters, e.g., ```json{{\"rooms\":[...]}}```.

Extract the following for EACH room mentioned:
- Room type (bedroom, kitchen, bathroom, living_room, hall, etc.)
- Dimensions (width and height in inches)
- Number of windows (default: 1 if not specified)
- Number of doors (default: 1 if not specified)

Default dimensions if not specified:
- Bedroom: 180x144
- Kitchen: 120x120
- Bathroom: 96x72
- Living room: 200x180
- Hall: 96x60

Output ONLY with this structure (a JSON object with a 'rooms' key containing a list of rooms, wrapped in ```json tags):
```json
{{
  "rooms": [
    {{
      "rtype": "bedroom",
      "id": 1,
      "width": 180,
      "height": 144,
      "windows": 2,
      "doors": 1
    }},
    {{
      "rtype": "kitchen",
      "id": 2,
      "width": 120,
      "height": 120,
      "windows": 1,
      "doors": 1
    }},
    {{
      "rtype": "bathroom",
      "id": 3,
      "width": 96,
      "height": 72,
      "windows": 1,
      "doors": 1
    }}
  ]
}}
```
Output:```json
""",
    partial_variables={"instructions_rooms_container": instructions_rooms_container}
)

position_prompt = PromptTemplate(
    input_variables=["rooms"],
    template="""You are an expert floor plan designer. Assign [x, y] coordinates to each room in the floor plan.
Consider the dimensions of each room to place them logically adjacent to each other.
Assume the origin [0,0] is the bottom-left corner of the overall floor plan.

Given Rooms (as JSON): {rooms}

Output Instructions: {instructions_rooms_container}

YOUR RESPONSE MUST CONTAIN ONLY THE JSON OBJECT AND NOTHING ELSE. DO NOT INCLUDE ANY CONVERSATIONAL TEXT OR EXPLANATIONS. Output the JSON object wrapped in markdown code delimiters, e.g., ```json{{\"rooms\":[...]}}```.
Output:```json
""",
    partial_variables={"instructions_rooms_container": instructions_rooms_container}
)

# ===================== STATE =====================
class FPState(TypedDict):
    user_input: str
    rooms: List[Dict]
    positioned_rooms: List[Dict]
    final: Dict

# ===================== NODES =====================
def extract_rooms(state: FPState):
    # Use RoomsContainer here
    parser = PydanticOutputParser(pydantic_object=RoomsContainer)
    chain = room_prompt | model | parser
    # The response will be a RoomsContainer object
    rooms_container = chain.invoke({"user_input": state["user_input"]})
    state["rooms"] = [r.model_dump() for r in rooms_container.rooms]
    return state

def position_rooms(state: FPState):
    # Use RoomsContainer for positioning as well, assuming it outputs a list of rooms
    parser = PydanticOutputParser(pydantic_object=RoomsContainer)
    chain = position_prompt | model | parser
    # The response will be a RoomsContainer object
    positioned_rooms_container = chain.invoke({"rooms": json.dumps(state["rooms"])})
    state["positioned_rooms"] = [r.model_dump() for r in positioned_rooms_container.rooms]
    return state

def add_furniture(state: FPState):
    for room in state["positioned_rooms"]:
        furn = []
        if room["rtype"] == "bedroom":
            f = FURNITURE_LIBRARY["wardrobe"]
            furn.append(Furniture(
                ftype="wardrobe",
                width=f["width"],
                height=f["height"],
                position=[10, 10]
            ))
        if room["rtype"] in ["kitchen", "bathroom"]:
            f = FURNITURE_LIBRARY["sink"]
            furn.append(Furniture(
                ftype="sink",
                width=f["width"],
                height=f["height"],
                position=[5, 5]
            ))
        room["furniture"] = [x.model_dump() for x in furn]
    return state

def finalize(state: FPState):
    area = sum(r["width"] * r["height"] for r in state["positioned_rooms"])
    state["final"] = FloorPlanSchema(
        rooms=[Room(**r) for r in state["positioned_rooms"]],
        connections=[],
        total_area=area
    ).model_dump(by_alias=True)
    return state

# ===================== WORKFLOW =====================
graph = StateGraph(FPState)
graph.add_node("extract", extract_rooms)
graph.add_node("position", position_rooms)
graph.add_node("furniture", add_furniture)
graph.add_node("final", finalize)

graph.set_entry_point("extract")
graph.add_edge("extract", "position")
graph.add_edge("position", "furniture")
graph.add_edge("furniture", "final")

app = graph.compile()

# ===================== RUN =====================
def main():
    text = input("Describe your floor plan: ")
    result = app.invoke({
        "user_input": text,
        "rooms": [],
        "positioned_rooms": [],
        "final": {}
    })
    print(json.dumps(result["final"], indent=2))

main()


Describe your floor plan: make a 2 BHK plan with 1 bathroom
{
  "rooms": [
    {
      "rtype": "bedroom",
      "id": 1,
      "width": 180,
      "height": 144,
      "position": [
        0,
        0
      ],
      "windows": 1,
      "doors": 1,
      "furniture": [
        {
          "ftype": "wardrobe",
          "width": 60,
          "height": 24,
          "position": [
            10,
            10
          ]
        }
      ]
    },
    {
      "rtype": "bedroom",
      "id": 2,
      "width": 180,
      "height": 144,
      "position": [
        180,
        0
      ],
      "windows": 1,
      "doors": 1,
      "furniture": [
        {
          "ftype": "wardrobe",
          "width": 60,
          "height": 24,
          "position": [
            10,
            10
          ]
        }
      ]
    },
    {
      "rtype": "hall",
      "id": 3,
      "width": 96,
      "height": 60,
      "position": [
        0,
        144
      ],
      "windows": 1,
      "doors":

In [ ]:
import re
import json
from typing import TypedDict, List, Dict

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langgraph.graph import StateGraph
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

# =========================================================
# AREA NORMALIZATION
# =========================================================

def normalize_area(user_input: str) -> int:
    """
    Converts user area input to square inches.
    Supports: sq ft, square feet, sq meter, sqm
    """
    match = re.search(r"(\d+)\s*(sq\s*ft|square\s*feet|sq\s*m|sqm|square\s*meter)", user_input.lower())
    if not match:
        return 0

    value = int(match.group(1))
    unit = match.group(2)

    if "ft" in unit:
        return value * 144
    if "m" in unit:
        return int(value * 1550)

    return 0

# =========================================================
# HELPER FOR ROBUST JSON EXTRACTION
# =========================================================

def extract_json_from_response(text: str) -> str:
    """Extracts a JSON string from a text that might contain markdown or other text."""
    # Pattern to find a markdown JSON block
    match = re.search(r"```json\s*(\{.*?})\s*```", text, re.DOTALL)
    if match:
        return match.group(1)
    # If no markdown block, try to find a standalone JSON object
    try:
        # Attempt to parse directly, assuming it might be just JSON
        json.loads(text)
        return text
    except json.JSONDecodeError:
        # Fallback to regex if direct parse fails and no markdown block was found
        match = re.search(r"(\{.*})", text, re.DOTALL)
        if match:
            return match.group(1)
    return "" # Return empty string if no JSON found

# =========================================================
# LLM SETUP
# =========================================================

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    max_new_tokens=2048,
    temperature=0.3,
)

model = ChatHuggingFace(llm=llm)

# =========================================================
# SCHEMAS
# =========================================================

class Furniture(BaseModel):
    ftype: str
    width: int
    height: int
    position: List[int]

class Room(BaseModel):
    rtype: str
    id: int
    width: int
    height: int
    position: List[int]
    windows: int
    doors: int
    furniture: List[Furniture] = []

class Connection(BaseModel):
    from_room: int = Field(alias="from")
    to_room: int = Field(alias="to")
    rtype: str

class FloorPlanSchema(BaseModel):
    rooms: List[Room]
    connections: List[Connection]
    total_area: int

# Existing Pydantic model for multiple rooms (from previous cell 'yeeYv4sk8DIJ')
class room_scheme(BaseModel):
    rtype: str = Field(description="Type of room (bedroom, kitchen, bathroom, living_room, hall, etc.)")
    id: int = Field(description="Unique identifier for the room, starting from 1")
    width: int = Field(description="Width of the room in inches")
    height: int = Field(description="Height/length of the room in inches")
    windows: int = Field(default=0, description="Number of windows in the room")
    doors: int = Field(default=1, description="Number of doors in the room")

class getrooms(BaseModel):
    rooms: List[room_scheme] = Field(description="List of all rooms in the floor plan")

parser_r = PydanticOutputParser(pydantic_object=getrooms)
instructions_r = parser_r.get_format_instructions()

# Existing Pydantic model for connections (from previous cell 'yeeYv4sk8DIJ')
class getconnections(BaseModel):
    connections: List[Connection] = Field(description="List of all Connections connecting different rooms in the floor plan")

parser_c = PydanticOutputParser(pydantic_object=getconnections)
instructions_c = parser_c.get_format_instructions()

# Existing Pydantic model for positioned rooms (from previous cell 'yeeYv4sk8DIJ')
class PositionedRooms(BaseModel):
    rooms: List[Room] = Field(description="List of rooms with assigned [x, y] positions")

parser_p = PydanticOutputParser(pydantic_object=PositionedRooms)
instructions_p = parser_p.get_format_instructions()

# =========================================================
# PROMPTS
# =========================================================

room_prompt = PromptTemplate(
    input_variables=["user_input"],
    template="""You are an expert floor plan analyzer. Extract all room information from the user's description.

User Input: {user_input}

Output Instructions: {instructions_r}

YOUR RESPONSE MUST CONTAIN ONLY THE JSON OBJECT AND NOTHING ELSE. DO NOT INCLUDE ANY CONVERSATIONAL TEXT OR EXPLANATIONS. Output the JSON object wrapped in markdown code delimiters, e.g., ```json{{\"rooms\":[...]}}```.

Extract the following for EACH room mentioned:
- Room type (bedroom, kitchen, bathroom, living_room, hall, etc.)
- Dimensions (width and height in inches)
- Number of windows (default: 1 if not specified)
- Number of doors (default: 1 if not specified)

Default dimensions if not specified:
- Bedroom: 180x144
- Kitchen: 120x120
- Bathroom: 96x72
- Living room: 200x180
- Hall: 96x60

Output ONLY with this structure (a JSON object with a 'rooms' key containing a list of rooms, wrapped in ```json tags):
```json
{{
  "rooms": [
    {{
      "rtype": "bedroom",
      "id": 1,
      "width": 180,
      "height": 144,
      "windows": 2,
      "doors": 1
    }},
    {{
      "rtype": "kitchen",
      "id": 2,
      "width": 120,
      "height": 120,
      "windows": 1,
      "doors": 1
    }},
    {{
      "rtype": "bathroom",
      "id": 3,
      "width": 96,
      "height": 72,
      "windows": 1,
      "doors": 1
    }}
  ]
}}
```
Output:```json
""",
    partial_variables={"instructions_r": instructions_r}
)

connection_prompt = PromptTemplate(
    input_variables=["user_input", "rooms"],
    template="""You are an expert floor plan analyzer. Extract all connections between rooms.

User Input: {user_input}

Available Rooms: {rooms}



Identify all relationships:
- "Room A connects to Room B" \u2192 door connection
- "Room A is adjacent to Room B" \u2192 door connection
- "Room A has a window facing Room B" \u2192 window connection
- "Room A has windows" \u2192 window connection to outside (from: room_id, to: room_id - same ID)

Output Instructions: {instructions_c}

Output ONLY with this structure (a JSON object with a 'connections' key containing a list of connections, wrapped in ```json tags):
```json
{{
  "connections": [
    {{
      "from": 1,
      "to": 2,
      "rtype": "door"
    }},
    {{
      "from": 1,
      "to": 1,
      "rtype": "window"
    }}
  ]
}}
```
Output:```json
""",
    partial_variables={"instructions_c": instructions_c}
)

position_prompt = PromptTemplate(
    input_variables=["extracted_rooms", "extracted_connections"],
    template="""You are an expert floor plan designer. Assign [x, y] coordinates to each room in the floor plan.
Consider the dimensions of each room and the connections between them to place them logically adjacent to each other.
Assume the origin [0,0] is the bottom-left corner of the overall floor plan.
Ensure rooms connected by a 'door' are adjacent. Windows can face outside, so they don't necessarily imply adjacency.

Extracted Rooms: {extracted_rooms}
Extracted Connections: {extracted_connections}

Output Instructions: {instructions_p}

Output ONLY with this structure (a JSON object with a 'rooms' key containing a list of rooms, wrapped in ```json tags):
```json
{{
  "rooms": [
    {{
      "rtype": "bedroom",
      "id": 1,
      "width": 180,
      "height": 144,
      "position": [0,0],
      "windows": 1,
      "doors": 1
    }}
  ]
}}
```
Output:```json
""",
    partial_variables={"instructions_p": instructions_p}
)

# =========================================================
# STATE
# =========================================================

class FloorState(TypedDict):
    user_input: str
    rooms: List[Dict]
    connections: List[Dict]
    positioned_rooms: List[Dict]
    final_output: Dict

# =========================================================
# FURNITURE RULE ENGINE (DETERMINISTIC)
# =========================================================

def add_furniture(rooms: List[Dict]) -> List[Dict]:
    for room in rooms:
        furn = []

        if room["rtype"] == "bedroom":
            furn.append({
                "ftype": "wardrobe",
                "width": 60,
                "height": 24,
                "position": [10, 10]
            })

        if room["rtype"] in ["kitchen", "bathroom"]:
            furn.append({
                "ftype": "sink",
                "width": 24,
                "height": 18,
                "position": [5, 5]
            })

        room["furniture"] = furn

    return rooms

# =========================================================
# WORKFLOW NODES
# =========================================================

def extract_rooms(state: FloorState):
    print("Extracting rooms...")
    llm_raw_response = model.invoke(room_prompt.format(user_input=state["user_input"])).content
    json_str = extract_json_from_response(llm_raw_response)
    llm_response_parsed = parser_r.parse(json_str)
    content = [room.model_dump() for room in llm_response_parsed.rooms]
    state["rooms"] = content if content else []
    print(f"✅ Rooms extracted successfully! Found {len(state['rooms'])} rooms")
    return state

def extract_connections(state: FloorState):
    print("🔗 Extracting connections...")
    llm_raw_response = model.invoke(connection_prompt.format(
        user_input=state["user_input"],
        rooms=json.dumps(state["rooms"])
    )).content
    json_str = extract_json_from_response(llm_raw_response)
    llm_response_parsed = parser_c.parse(json_str)
    content = [conn.model_dump(by_alias=True) for conn in llm_response_parsed.connections]
    state["connections"] = content if content else []
    print(f"✅ Connections extracted successfully! Found {len(state['connections'])} connections")
    return state

def position_rooms(state: FloorState):
    print("Positioning rooms...")
    llm_raw_response = model.invoke(position_prompt.format(
        extracted_rooms=json.dumps(state["rooms"]),
        extracted_connections=json.dumps(state["connections"])
    )).content
    json_str = extract_json_from_response(llm_raw_response)
    llm_response_parsed = parser_p.parse(json_str)
    state["positioned_rooms"] = [room.model_dump() for room in llm_response_parsed.rooms]
    print(f"✅ Rooms positioned successfully! Positioned {len(state['positioned_rooms'])} rooms")
    return state

def finalize(state: FloorState):
    print("Formatting final output...")
    rooms_with_furniture = add_furniture(state["positioned_rooms"])

    # Calculate total area
    total_area = sum(r["width"] * r["height"] for r in rooms_with_furniture)

    final = FloorPlanSchema(
        rooms=[Room(**r) for r in rooms_with_furniture],
        connections=[Connection(**c) for c in state["connections"]],
        total_area=total_area
    )

    state["final_output"] = final.model_dump(by_alias=True)
    print("✅ Final output formatted successfully!")
    return state

# =========================================================
# GRAPH
# =========================================================

graph = StateGraph(FloorState)
graph.add_node("rooms", extract_rooms)
graph.add_node("connections", extract_connections)
graph.add_node("position", position_rooms)
graph.add_node("final", finalize)

graph.set_entry_point("rooms")
graph.add_edge("rooms", "connections")
graph.add_edge("connections", "position")
graph.add_edge("position", "final")

app = graph.compile()

# =========================================================
# RUN
# =========================================================

def main():
    user_input = input("Describe your floor plan: ")
    # area = normalize_area(user_input) # Not directly used in the workflow for now

    result = app.invoke({
        "user_input": user_input,
        "rooms": [],
        "connections": [],
        "positioned_rooms": [],
        "final_output": {}
    })

    print(json.dumps(result["final_output"], indent=2))

if __name__ == "__main__":
    main()

Describe your floor plan: 1200 sq ft 2 BHK 
Extracting rooms...
✅ Rooms extracted successfully! Found 4 rooms
🔗 Extracting connections...
✅ Connections extracted successfully! Found 7 connections
Positioning rooms...
✅ Rooms positioned successfully! Positioned 4 rooms
Formatting final output...
✅ Final output formatted successfully!
{
  "rooms": [
    {
      "rtype": "bedroom",
      "id": 1,
      "width": 180,
      "height": 144,
      "position": [
        -180,
        0
      ],
      "windows": 1,
      "doors": 1,
      "furniture": [
        {
          "ftype": "wardrobe",
          "width": 60,
          "height": 24,
          "position": [
            10,
            10
          ]
        }
      ]
    },
    {
      "rtype": "bedroom",
      "id": 2,
      "width": 180,
      "height": 144,
      "position": [
        96,
        0
      ],
      "windows": 1,
      "doors": 1,
      "furniture": [
        {
          "ftype": "wardrobe",
          "width": 60,
        

15/12

In [ ]:
import json

user_input = "Generate a 120 square meter 2 bhk plan with a hall and kitchen area."

print("🚀 Starting floor plan generation...")
result = app.invoke({"user_input": user_input})

print("\n✨ Final Output:\n")
print(json.dumps(result["final_output"], indent=2))

output_filename = "floor_plan.json"
with open(output_filename, "w") as f:
    json.dump(result["final_output"], f, indent=2)

print(f"✅ Floor plan saved to {output_filename}")
print("\n✅ Workflow execution complete!")

🚀 Starting floor plan generation...
Extracting rooms...
Chain is Ready (raw output mode)
Response Invoked
✅ Rooms extracted successfully! Found 4 rooms
🔗 Extracting connections...
✅ Connections extracted successfully! Found 7 connections
Positioning rooms...
✅ Rooms positioned successfully! Positioned 4 rooms
Formatting final output...
✅ Final output formatted successfully!

✨ Final Output:

{
  "rooms": [
    {
      "rtype": "bedroom",
      "id": 1,
      "width": 180,
      "height": 144,
      "position": [
        0,
        0
      ],
      "windows": 1,
      "doors": 1
    },
    {
      "rtype": "bedroom",
      "id": 2,
      "width": 180,
      "height": 144,
      "position": [
        276,
        0
      ],
      "windows": 1,
      "doors": 1
    },
    {
      "rtype": "kitchen",
      "id": 3,
      "width": 120,
      "height": 120,
      "position": [
        180,
        60
      ],
      "windows": 1,
      "doors": 1
    },
    {
      "rtype": "hall",
      "id"